In [43]:
!pip install stable-baselines3 gymnasium

In [44]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class GridWorldEnv(gym.Env):
    def __init__(self, grid_size=5, n_obstacles=3):
        super().__init__()
        self.grid_size = grid_size
        self.n_obstacles = n_obstacles
        self.action_space = spaces.Discrete(4)

        # 3 отдельных "слоя": позиция агента, позиция цели, препятствия
        # каждый слой — своя сетка из 0 и 1 (one-hot), без ложного порядка чисел
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(3 * grid_size * grid_size,), dtype=np.float32
        )
        self.max_steps = 50

    def _get_obs(self):
        agent_layer = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        goal_layer = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        obstacle_layer = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)

        agent_layer[self.agent_pos[1], self.agent_pos[0]] = 1
        goal_layer[self.goal_pos[1], self.goal_pos[0]] = 1
        for obs_pos in self.obstacles:
            obstacle_layer[obs_pos[1], obs_pos[0]] = 1

        return np.concatenate([
            agent_layer.flatten(), goal_layer.flatten(), obstacle_layer.flatten()
        ])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = np.array([0, 0])
        self.goal_pos = np.array([self.grid_size - 1, self.grid_size - 1])

        self.obstacles = []
        while len(self.obstacles) < self.n_obstacles:
            pos = self.np_random.integers(0, self.grid_size, size=2)
            if not np.array_equal(pos, self.agent_pos) and not np.array_equal(pos, self.goal_pos):
                if not any(np.array_equal(pos, o) for o in self.obstacles):
                    self.obstacles.append(pos)

        self.steps = 0
        return self._get_obs(), {}

    def step(self, action):
        self.steps += 1
        old_pos = self.agent_pos.copy()
        new_pos = self.agent_pos.copy()

        if action == 0:
            new_pos[1] = min(self.grid_size - 1, new_pos[1] + 1)
        elif action == 1:
            new_pos[1] = max(0, new_pos[1] - 1)
        elif action == 2:
            new_pos[0] = max(0, new_pos[0] - 1)
        elif action == 3:
            new_pos[0] = min(self.grid_size - 1, new_pos[0] + 1)

        hit_obstacle = any(np.array_equal(new_pos, obs) for obs in self.obstacles)
        hit_wall = np.array_equal(new_pos, old_pos)  # упёрся в границу — позиция не изменилась

        if not hit_obstacle:
            self.agent_pos = new_pos

        reached_goal = np.array_equal(self.agent_pos, self.goal_pos)

        if reached_goal:
            reward = 10.0
        elif hit_obstacle:
            reward = -5.0
        elif hit_wall:
            reward = -1.0  # отдельный, заметный штраф именно за удар в границу поля
        else:
            old_dist = np.linalg.norm(old_pos - self.goal_pos)
            new_dist = np.linalg.norm(self.agent_pos - self.goal_pos)
            reward = -0.1 + 0.5 * (old_dist - new_dist)

        terminated = reached_goal
        truncated = self.steps >= self.max_steps
        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        grid = np.full((self.grid_size, self.grid_size), '.')
        for obs_pos in self.obstacles:
            grid[obs_pos[1], obs_pos[0]] = 'X'
        grid[self.goal_pos[1], self.goal_pos[0]] = 'G'
        grid[self.agent_pos[1], self.agent_pos[0]] = 'A'
        print('\n'.join(' '.join(row) for row in grid[::-1]))
        print()

In [45]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

env = GridWorldEnv(grid_size=5, n_obstacles=3)
check_env(env)

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-3,
    buffer_size=20000,
    learning_starts=1000,
    policy_kwargs=dict(net_arch=[128, 128]),  # сеть побольше, раз вход теперь 75 чисел, а не 25
    verbose=1
)

model.learn(total_timesteps=150000)
model.save("gridworld_dqn")

Выходные данные были обрезаны до нескольких последних строк (5000).
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.0283   |
|    n_updates        | 33097    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 12.4     |
|    ep_rew_mean      | 9.59     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 7744     |
|    fps              | 797      |
|    time_elapsed     | 167      |
|    total_timesteps  | 133506   |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.00692  |
|    n_updates        | 33126    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 11.8     |
|    ep_rew_mean      | 10.3     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         

In [46]:
successes = 0
n_trials = 50

for trial in range(n_trials):
    obs, info = env.reset()
    for step in range(50):
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated:
            successes += 1
            break
        if truncated:
            break

print(f"Успешно дошёл до цели: {successes}/{n_trials} раз ({successes/n_trials*100:.1f}%)")

Успешно дошёл до цели: 43/50 раз (86.0%)


In [47]:
obs, info = env.reset()
env.render()
for step in range(20):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()
    print(f"Шаг {step}: действие={action}, награда={reward:.2f}")
    if terminated or truncated:
        break

X . . . G
. . . . .
. X . . .
. . . . .
A . . . X

X . . . G
. . . . .
. X . . .
A . . . .
. . . . X

Шаг 0: действие=0, награда=0.23
X . . . G
. . . . .
. X . . .
. A . . .
. . . . X

Шаг 1: действие=3, награда=0.28
X . . . G
. . . . .
. X . . .
. . A . .
. . . . X

Шаг 2: действие=3, награда=0.22
X . . . G
. . . . .
. X A . .
. . . . .
. . . . X

Шаг 3: действие=0, награда=0.29
X . . . G
. . A . .
. X . . .
. . . . .
. . . . X

Шаг 4: действие=0, награда=0.20
X . A . G
. . . . .
. X . . .
. . . . .
. . . . X

Шаг 5: действие=0, награда=0.02
X . . A G
. . . . .
. X . . .
. . . . .
. . . . X

Шаг 6: действие=3, награда=0.40
X . . . A
. . . . .
. X . . .
. . . . .
. . . . X

Шаг 7: действие=3, награда=10.00
